# 21 — Evaluation Metrics and Model Analysis

In the previous notebook, we studied advanced PyTorch training techniques.

Now we will focus on a question that is just as important as training:

> **How do we know whether a classifier is actually good?**

A single number such as accuracy is often not enough.

This becomes especially important when:

- Classes are imbalanced
- False negatives and false positives have different costs
- We need to choose a decision threshold
- We are working with medical imaging
- We want to compare models fairly
- We need to understand *where* the model fails

## In this notebook, we will study:

1. Why accuracy is not enough
2. Confusion matrices
3. True positives, false positives, true negatives, false negatives
4. Precision
5. Recall / sensitivity
6. Specificity
7. F1 score
8. Class imbalance
9. ROC curves
10. AUROC
11. Precision-recall curves
12. AUPRC
13. Threshold selection
14. Multi-class metrics
15. Per-class analysis
16. Calibration intuition
17. Error analysis
18. Choosing metrics for medical imaging
19. Common evaluation mistakes
20. Practice exercises

## Main Goal

By the end of this notebook, you should be able to move from:

$$
\boxed{
\text{Model Output}
}
$$

to:

$$
\boxed{
\text{Probabilities}
\rightarrow
\text{Threshold / Predicted Class}
\rightarrow
\text{Confusion Matrix}
\rightarrow
\text{Metrics}
\rightarrow
\text{Error Analysis}
}
$$

The most important principle is:

> **Choose metrics based on the real objective of the task, not because a metric is popular.**


In [ ]:
import math
import torch
import matplotlib.pyplot as plt

print("PyTorch version:", torch.__version__)


# 1. Classification Evaluation Starts With Predictions

A trained classifier usually produces:

> **Logits**

For binary classification with one output logit per sample:

$$
logit
\rightarrow
sigmoid
\rightarrow
probability
$$

For multi-class classification:

$$
logits
\rightarrow
argmax
\rightarrow
predicted\ class
$$

The metric is calculated from:

- True targets
- Predicted classes
- Or predicted probabilities/scores


# 2. Binary Classification Example

We will begin with a small binary classification example.

Target meanings:

$$
\begin{array}{|c|c|}
\hline
0 & Negative \\
\hline
1 & Positive \\
\hline
\end{array}
$$

Suppose our model outputs probabilities of the positive class.


In [ ]:
targets = torch.tensor([
    1, 1, 1, 1, 1,
    0, 0, 0, 0, 0
])

probabilities = torch.tensor([
    0.95,
    0.82,
    0.70,
    0.55,
    0.40,
    0.80,
    0.62,
    0.35,
    0.20,
    0.05
])

print("Targets:", targets)
print("Probabilities:", probabilities)


# 3. Converting Probabilities Into Classes

A threshold converts probability into a class prediction.

For threshold:

$$
t=0.5
$$

we predict:

$$
\hat{y}=
\begin{cases}
1 & p\geq0.5\\
0 & p<0.5
\end{cases}
$$


In [ ]:
threshold = 0.5

predictions = (
    probabilities
    >= threshold
).long()

print("Predictions:", predictions)


# 4. Why Accuracy Is Not Enough

Accuracy is:

$$
\boxed{
Accuracy
=
\frac{
Correct\ Predictions
}{
Total\ Predictions
}
}
$$

It answers:

> What fraction of all predictions were correct?

This is useful, but it does not tell us:

- Which class was missed
- Whether errors were false positives or false negatives
- Whether minority-class performance is poor
- Whether the decision threshold is appropriate


In [ ]:
accuracy = (
    predictions
    == targets
).float().mean()

print(
    "Accuracy:",
    accuracy.item()
)


# 5. The Class-Imbalance Problem

Suppose:

$$
95\%
$$

of samples are negative.

A useless model that always predicts negative gets:

$$
95\%
$$

accuracy.

That looks high, but the model detects:

$$
0\%
$$

of positive cases.

This is why class imbalance makes accuracy potentially misleading.


In [ ]:
imbalanced_targets = torch.cat([
    torch.zeros(
        95,
        dtype=torch.long
    ),
    torch.ones(
        5,
        dtype=torch.long
    )
])

always_negative = torch.zeros_like(
    imbalanced_targets
)

imbalanced_accuracy = (
    always_negative
    == imbalanced_targets
).float().mean()

print(
    "Accuracy:",
    imbalanced_accuracy.item()
)


# 6. Confusion Matrix

A binary confusion matrix separates predictions into four categories:

$$
\begin{array}{c|c|c}
 & \textbf{Predicted Positive} & \textbf{Predicted Negative} \\
\hline
\textbf{Actual Positive} & TP & FN \\
\hline
\textbf{Actual Negative} & FP & TN \\
\end{array}
$$

These four counts are the foundation for many classification metrics.


# 7. True Positive — TP

A **true positive** means:

- Actual class = positive
- Prediction = positive

$$
\boxed{
y=1,\ \hat{y}=1
}
$$

Example in medical imaging:

> Disease is present, and the model correctly predicts disease.


# 8. False Positive — FP

A **false positive** means:

- Actual class = negative
- Prediction = positive

$$
\boxed{
y=0,\ \hat{y}=1
}
$$

Medical example:

> Disease is absent, but the model predicts disease.


# 9. True Negative — TN

A **true negative** means:

- Actual class = negative
- Prediction = negative

$$
\boxed{
y=0,\ \hat{y}=0
}
$$


# 10. False Negative — FN

A **false negative** means:

- Actual class = positive
- Prediction = negative

$$
\boxed{
y=1,\ \hat{y}=0
}
$$

Medical example:

> Disease is present, but the model misses it.


# 11. Computing TP, FP, TN, and FN


In [ ]:
def binary_confusion_counts(
    targets,
    predictions
):
    targets = targets.long()
    predictions = predictions.long()

    tp = (
        (targets == 1)
        & (predictions == 1)
    ).sum().item()

    fp = (
        (targets == 0)
        & (predictions == 1)
    ).sum().item()

    tn = (
        (targets == 0)
        & (predictions == 0)
    ).sum().item()

    fn = (
        (targets == 1)
        & (predictions == 0)
    ).sum().item()

    return tp, fp, tn, fn

tp, fp, tn, fn = (
    binary_confusion_counts(
        targets,
        predictions
    )
)

print("TP:", tp)
print("FP:", fp)
print("TN:", tn)
print("FN:", fn)


# 12. Accuracy From the Confusion Matrix

Accuracy can be written as:

$$
\boxed{
Accuracy
=
\frac{
TP+TN
}{
TP+TN+FP+FN
}
}
$$


In [ ]:
accuracy_from_counts = (
    (tp + tn)
    / (tp + tn + fp + fn)
)

print(
    "Accuracy:",
    accuracy_from_counts
)


# 13. Precision

Precision asks:

> Of the samples predicted as positive, how many were actually positive?

$$
\boxed{
Precision
=
\frac{
TP
}{
TP+FP
}
}
$$

High precision means relatively few false positives.


In [ ]:
precision = (
    tp
    / (tp + fp)
    if (tp + fp) > 0
    else 0.0
)

print(
    "Precision:",
    precision
)


# 14. Recall / Sensitivity

Recall asks:

> Of all actual positive samples, how many did the model detect?

$$
\boxed{
Recall
=
Sensitivity
=
\frac{
TP
}{
TP+FN
}
}
$$

High recall means relatively few false negatives.


In [ ]:
recall = (
    tp
    / (tp + fn)
    if (tp + fn) > 0
    else 0.0
)

print(
    "Recall / Sensitivity:",
    recall
)


# 15. Specificity

Specificity asks:

> Of all actual negative samples, how many did the model correctly reject?

$$
\boxed{
Specificity
=
\frac{
TN
}{
TN+FP
}
}
$$

High specificity means relatively few false positives.


In [ ]:
specificity = (
    tn
    / (tn + fp)
    if (tn + fp) > 0
    else 0.0
)

print(
    "Specificity:",
    specificity
)


# 16. Precision and Recall Measure Different Things

$$
\begin{array}{|c|c|}
\hline
\textbf{Metric} & \textbf{Question} \\
\hline
Precision &
\text{When model predicts positive, how often is it correct?} \\
\hline
Recall &
\text{How many real positives did the model find?} \\
\hline
Specificity &
\text{How many real negatives did the model correctly reject?} \\
\hline
\end{array}
$$


# 17. F1 Score

The F1 score combines precision and recall using the harmonic mean:

$$
\boxed{
F1
=
2
\cdot
\frac{
Precision\cdot Recall
}{
Precision+Recall
}
}
$$

Equivalent form:

$$
\boxed{
F1
=
\frac{
2TP
}{
2TP+FP+FN
}
}
$$


In [ ]:
f1 = (
    2
    * precision
    * recall
    / (precision + recall)
    if (precision + recall) > 0
    else 0.0
)

print(
    "F1:",
    f1
)


# 18. Why Harmonic Mean?

The harmonic mean becomes small when one of the two values is small.

For example:

$$
Precision=0.95
$$

but:

$$
Recall=0.20
$$

does not produce a high F1.

So F1 rewards models that balance precision and recall.


# 19. A Reusable Binary-Metrics Function


In [ ]:
def binary_metrics(
    targets,
    probabilities,
    threshold=0.5
):
    predictions = (
        probabilities
        >= threshold
    ).long()

    tp, fp, tn, fn = (
        binary_confusion_counts(
            targets,
            predictions
        )
    )

    total = (
        tp + fp + tn + fn
    )

    accuracy = (
        (tp + tn)
        / total
        if total > 0
        else 0.0
    )

    precision = (
        tp
        / (tp + fp)
        if (tp + fp) > 0
        else 0.0
    )

    recall = (
        tp
        / (tp + fn)
        if (tp + fn) > 0
        else 0.0
    )

    specificity = (
        tn
        / (tn + fp)
        if (tn + fp) > 0
        else 0.0
    )

    f1 = (
        2
        * precision
        * recall
        / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )

    return {
        "threshold": threshold,
        "tp": tp,
        "fp": fp,
        "tn": tn,
        "fn": fn,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "sensitivity": recall,
        "specificity": specificity,
        "f1": f1
    }

metrics = binary_metrics(
    targets,
    probabilities,
    threshold=0.5
)

print(metrics)


# 20. Threshold Selection Changes the Metrics

The threshold is not merely a technical detail.

Changing the threshold changes:

- TP
- FP
- TN
- FN
- Precision
- Recall
- Specificity
- F1

For example:

$$
threshold\downarrow
$$

usually produces more positive predictions.


In [ ]:
for threshold in [
    0.3,
    0.5,
    0.7
]:
    metrics = binary_metrics(
        targets,
        probabilities,
        threshold=threshold
    )

    print(
        f"Threshold {threshold:.1f} | "
        f"Precision {metrics['precision']:.3f} | "
        f"Recall {metrics['recall']:.3f} | "
        f"Specificity {metrics['specificity']:.3f} | "
        f"F1 {metrics['f1']:.3f}"
    )


# 21. Lower Threshold

Lowering the threshold generally:

- Increases positive predictions
- Tends to increase sensitivity
- Tends to decrease specificity
- Can increase false positives

This is a tradeoff, not automatically an improvement.


# 22. Higher Threshold

Raising the threshold generally:

- Decreases positive predictions
- Tends to increase specificity
- Tends to decrease sensitivity
- Can increase false negatives

The best threshold depends on the application.


# 23. Thresholds Should Be Chosen on Validation Data

Do not choose the threshold using the test set.

Correct workflow:

$$
\boxed{
Train
\rightarrow
Choose\ Threshold\ on\ Validation
\rightarrow
Evaluate\ Once\ on\ Test
}
$$

Otherwise the test set becomes part of model development.


# 24. ROC Curve

ROC stands for:

> **Receiver Operating Characteristic**

The ROC curve examines classifier behavior over many thresholds.

The axes are:

$$
TPR
=
Sensitivity
=
\frac{TP}{TP+FN}
$$

and:

$$
FPR
=
\frac{FP}{FP+TN}
=
1-Specificity
$$


# 25. Computing ROC Points Manually

We will evaluate many thresholds.


In [ ]:
def roc_curve_torch(
    targets,
    scores
):
    thresholds = torch.cat([
        torch.tensor([
            float("inf")
        ]),
        torch.sort(
            torch.unique(scores),
            descending=True
        ).values,
        torch.tensor([
            float("-inf")
        ])
    ])

    fprs = []
    tprs = []

    for threshold in thresholds:
        predictions = (
            scores
            >= threshold
        ).long()

        tp, fp, tn, fn = (
            binary_confusion_counts(
                targets,
                predictions
            )
        )

        tpr = (
            tp
            / (tp + fn)
            if (tp + fn) > 0
            else 0.0
        )

        fpr = (
            fp
            / (fp + tn)
            if (fp + tn) > 0
            else 0.0
        )

        tprs.append(tpr)
        fprs.append(fpr)

    return (
        torch.tensor(fprs),
        torch.tensor(tprs),
        thresholds
    )

fpr, tpr, roc_thresholds = (
    roc_curve_torch(
        targets,
        probabilities
    )
)

print(
    "ROC points:",
    len(fpr)
)


# 26. Plotting the ROC Curve


In [ ]:
plt.figure(figsize=(6, 6))

plt.plot(
    fpr.numpy(),
    tpr.numpy(),
    marker="o",
    label="Model"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random baseline"
)

plt.xlabel(
    "False Positive Rate"
)

plt.ylabel(
    "True Positive Rate / Sensitivity"
)

plt.title(
    "ROC Curve"
)

plt.legend()
plt.show()


# 27. Interpreting the ROC Curve

A useful classifier tends toward the upper-left region:

$$
FPR\rightarrow0
$$

$$
TPR\rightarrow1
$$

A random classifier tends toward the diagonal.

ROC curves compare ranking behavior across thresholds.


# 28. AUROC

AUROC means:

> **Area Under the ROC Curve**

Conceptually:

$$
\boxed{
AUROC
\in
[0,1]
}
$$

Typical interpretation:

$$
\begin{array}{|c|c|}
\hline
AUROC\approx0.5 & \text{Random ranking} \\
\hline
AUROC\rightarrow1 & \text{Better separation} \\
\hline
\end{array}
$$

AUROC is threshold-independent in the sense that it summarizes performance across many thresholds.


# 29. Trapezoidal Area

We can approximate the ROC area with the trapezoidal rule.


In [ ]:
def trapezoid_area(
    x,
    y
):
    order = torch.argsort(
        x
    )

    x = x[order]
    y = y[order]

    dx = (
        x[1:]
        - x[:-1]
    )

    average_height = (
        y[1:]
        + y[:-1]
    ) / 2

    return (
        dx
        * average_height
    ).sum().item()

auroc = trapezoid_area(
    fpr,
    tpr
)

print(
    "AUROC:",
    auroc
)


# 30. AUROC Is a Ranking Metric

An important intuition is:

> AUROC measures how well the model ranks positive samples above negative samples.

It does not directly tell you:

- The final operating threshold
- The final precision
- The final sensitivity
- Calibration quality


# 31. ROC Can Look Good Under Severe Class Imbalance

The false-positive rate divides by:

$$
FP+TN
$$

If there are enormous numbers of negatives, even many false positives may correspond to a small FPR.

This is one reason precision-recall curves can be especially useful for rare-positive problems.


# 32. Precision-Recall Curve

A precision-recall curve plots:

$$
Precision
$$

against:

$$
Recall
$$

across thresholds.

It focuses strongly on positive-class performance.


In [ ]:
def precision_recall_curve_torch(
    targets,
    scores
):
    thresholds = torch.sort(
        torch.unique(scores),
        descending=True
    ).values

    precisions = []
    recalls = []

    for threshold in thresholds:
        predictions = (
            scores
            >= threshold
        ).long()

        tp, fp, tn, fn = (
            binary_confusion_counts(
                targets,
                predictions
            )
        )

        precision = (
            tp
            / (tp + fp)
            if (tp + fp) > 0
            else 1.0
        )

        recall = (
            tp
            / (tp + fn)
            if (tp + fn) > 0
            else 0.0
        )

        precisions.append(
            precision
        )

        recalls.append(
            recall
        )

    precisions = torch.tensor(
        precisions
    )

    recalls = torch.tensor(
        recalls
    )

    return (
        precisions,
        recalls,
        thresholds
    )

precisions, recalls, pr_thresholds = (
    precision_recall_curve_torch(
        targets,
        probabilities
    )
)

print(
    "PR points:",
    len(precisions)
)


# 33. Plotting the Precision-Recall Curve


In [ ]:
positive_prevalence = (
    targets.float().mean().item()
)

plt.figure(figsize=(6, 6))

plt.plot(
    recalls.numpy(),
    precisions.numpy(),
    marker="o",
    label="Model"
)

plt.axhline(
    positive_prevalence,
    linestyle="--",
    label="Prevalence baseline"
)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title(
    "Precision-Recall Curve"
)
plt.legend()
plt.show()


# 34. Precision-Recall Baseline

For an uninformative classifier, expected precision is related to positive prevalence.

If positive prevalence is:

$$
10\%
$$

the baseline precision is roughly:

$$
0.10
$$

This makes PR curves very interpretable in imbalanced problems.


# 35. AUPRC

AUPRC means:

> **Area Under the Precision-Recall Curve**

Higher is generally better.

Unlike AUROC, the baseline depends on positive prevalence.

This means AUPRC values are not directly comparable across datasets with very different prevalence without context.


# 36. Approximate AUPRC

We can sort points by recall and integrate with the trapezoidal rule for educational purposes.


In [ ]:
auprc = trapezoid_area(
    recalls,
    precisions
)

print(
    "Approximate AUPRC:",
    auprc
)


# 37. AUROC vs AUPRC

$$
\begin{array}{|c|c|}
\hline
\textbf{AUROC} & \textbf{AUPRC} \\
\hline
Uses\ TPR\ and\ FPR & Uses\ Precision\ and\ Recall \\
\hline
Useful\ ranking\ summary & Focuses\ on\ positive\ detection \\
\hline
Can\ look\ optimistic\ with\ rare\ positives & Often\ informative\ for\ imbalance \\
\hline
Baseline\approx0.5 & Baseline\approx prevalence \\
\hline
\end{array}
$$


# 38. Threshold Selection Strategies

There is no universally correct threshold.

Possible objectives include:

- Maximize F1
- Maximize sensitivity subject to minimum specificity
- Maximize specificity subject to minimum sensitivity
- Minimize clinical cost
- Use a threshold chosen from operational constraints

Threshold choice is part of the deployment objective.


# 39. Searching for Maximum F1


In [ ]:
candidate_thresholds = torch.linspace(
    0.0,
    1.0,
    101
)

best_threshold = None
best_f1 = -1.0

for threshold in candidate_thresholds:
    metrics = binary_metrics(
        targets,
        probabilities,
        threshold=float(
            threshold
        )
    )

    if metrics["f1"] > best_f1:
        best_f1 = metrics["f1"]
        best_threshold = float(
            threshold
        )

print(
    "Best threshold:",
    best_threshold
)

print(
    "Best F1:",
    best_f1
)


# 40. Sensitivity-Constrained Threshold Selection

In screening, we may require:

$$
Sensitivity\geq0.90
$$

and among those thresholds choose one with the highest specificity.

This expresses the real priority more directly than blindly maximizing accuracy.


In [ ]:
def choose_threshold_for_min_sensitivity(
    targets,
    probabilities,
    min_sensitivity=0.9
):
    candidates = []

    for threshold in torch.linspace(
        0.0,
        1.0,
        1001
    ):
        metrics = binary_metrics(
            targets,
            probabilities,
            threshold=float(
                threshold
            )
        )

        if (
            metrics["sensitivity"]
            >= min_sensitivity
        ):
            candidates.append(
                metrics
            )

    if not candidates:
        return None

    return max(
        candidates,
        key=lambda item:
            item["specificity"]
    )

selected = (
    choose_threshold_for_min_sensitivity(
        targets,
        probabilities,
        min_sensitivity=0.8
    )
)

print(selected)


# 41. Specificity-Constrained Threshold Selection

Another task may require:

$$
Specificity\geq0.95
$$

and then maximize sensitivity.

The correct rule depends on the real cost of errors.


# 42. Threshold Selection Must Be Predefined

A strong evaluation protocol defines:

- Which validation metric selects the model
- Which validation objective selects the threshold
- Which final metrics are reported on the test set

Do not choose the best-looking threshold after seeing test results.


# 43. Logits vs Probabilities for Ranking Metrics

ROC and PR curves require a **score** that ranks examples.

You can often use:

- Logits
- Probabilities

For sigmoid:

$$
sigmoid(x)
$$

is monotonic.

Therefore logits and sigmoid probabilities produce the same ranking and typically the same ROC/AUROC.

Probabilities are easier to interpret as probabilities.


# 44. Binary Logit Example


In [ ]:
binary_logits = torch.tensor([
    2.0,
    1.2,
    0.4,
    -0.2,
    -1.0
])

binary_probabilities = torch.sigmoid(
    binary_logits
)

print(
    binary_probabilities
)


# 45. Multi-Class Classification

Suppose there are:

$$
C=4
$$

classes.

Model output:

$$
(N,\ 4)
$$

Predicted class:

```python
logits.argmax(dim=1)
```

Targets:

$$
(N)
$$


In [ ]:
torch.manual_seed(42)

multiclass_logits = torch.tensor([
    [3.0, 0.2, 0.1],
    [0.1, 2.5, 0.4],
    [0.2, 0.5, 2.0],
    [1.5, 1.2, 0.3],
    [0.4, 2.1, 1.8],
    [2.0, 0.3, 0.1],
    [0.2, 0.4, 2.4],
    [0.8, 1.6, 0.7]
])

multiclass_targets = torch.tensor([
    0,
    1,
    2,
    1,
    2,
    0,
    2,
    1
])

multiclass_predictions = (
    multiclass_logits.argmax(
        dim=1
    )
)

print(
    "Predictions:",
    multiclass_predictions
)


# 46. Multi-Class Confusion Matrix

For:

$$
C
$$

classes, the confusion matrix has shape:

$$
\boxed{
(C,\ C)
}
$$

Rows:

> True class

Columns:

> Predicted class


In [ ]:
def confusion_matrix_multiclass(
    targets,
    predictions,
    num_classes
):
    matrix = torch.zeros(
        num_classes,
        num_classes,
        dtype=torch.long
    )

    for true_label, pred_label in zip(
        targets,
        predictions
    ):
        matrix[
            true_label.long(),
            pred_label.long()
        ] += 1

    return matrix

multi_confusion = (
    confusion_matrix_multiclass(
        multiclass_targets,
        multiclass_predictions,
        num_classes=3
    )
)

print(
    multi_confusion
)


# 47. Reading a Multi-Class Confusion Matrix

Diagonal entries are correct predictions:

$$
matrix[i,i]
$$

Off-diagonal entries represent class confusion.

Example:

$$
matrix[1,2]
$$

means:

> True class 1 was predicted as class 2.


# 48. Visualizing the Multi-Class Confusion Matrix


In [ ]:
plt.figure(figsize=(6, 5))

plt.imshow(
    multi_confusion.numpy()
)

plt.xlabel(
    "Predicted Class"
)

plt.ylabel(
    "True Class"
)

plt.title(
    "Multi-Class Confusion Matrix"
)

plt.xticks(
    range(3)
)

plt.yticks(
    range(3)
)

for row in range(3):
    for col in range(3):
        plt.text(
            col,
            row,
            int(
                multi_confusion[
                    row,
                    col
                ]
            ),
            ha="center",
            va="center"
        )

plt.colorbar()
plt.show()


# 49. Per-Class Metrics With One-vs-Rest

For one class:

> Treat that class as positive and all other classes as negative.

Then compute:

- TP
- FP
- TN
- FN
- Precision
- Recall
- Specificity
- F1

This gives per-class analysis.


In [ ]:
def per_class_counts(
    confusion,
    class_index
):
    tp = confusion[
        class_index,
        class_index
    ].item()

    fn = (
        confusion[
            class_index,
            :
        ].sum().item()
        - tp
    )

    fp = (
        confusion[
            :,
            class_index
        ].sum().item()
        - tp
    )

    total = confusion.sum().item()

    tn = (
        total
        - tp
        - fn
        - fp
    )

    return tp, fp, tn, fn

for class_index in range(3):
    print(
        class_index,
        per_class_counts(
            multi_confusion,
            class_index
        )
    )


# 50. Per-Class Metric Function


In [ ]:
def per_class_metrics(
    confusion
):
    results = []

    num_classes = (
        confusion.shape[0]
    )

    for class_index in range(
        num_classes
    ):
        tp, fp, tn, fn = (
            per_class_counts(
                confusion,
                class_index
            )
        )

        precision = (
            tp
            / (tp + fp)
            if (tp + fp) > 0
            else 0.0
        )

        recall = (
            tp
            / (tp + fn)
            if (tp + fn) > 0
            else 0.0
        )

        specificity = (
            tn
            / (tn + fp)
            if (tn + fp) > 0
            else 0.0
        )

        f1 = (
            2
            * precision
            * recall
            / (precision + recall)
            if (precision + recall) > 0
            else 0.0
        )

        support = (
            tp + fn
        )

        results.append({
            "class": class_index,
            "precision": precision,
            "recall": recall,
            "specificity": specificity,
            "f1": f1,
            "support": support
        })

    return results

class_results = (
    per_class_metrics(
        multi_confusion
    )
)

for item in class_results:
    print(item)


# 51. Support

Support means:

> Number of true samples belonging to a class.

For class $i$:

$$
support_i
=
TP_i+FN_i
$$

Support is important when classes are imbalanced.


# 52. Macro Averaging

Macro averaging gives every class equal weight.

For recall:

$$
\boxed{
Recall_{macro}
=
\frac{
1
}{
C
}
\sum_{i=1}^{C}
Recall_i
}
$$

A rare class matters just as much as a common class.


In [ ]:
macro_f1 = sum(
    item["f1"]
    for item in class_results
) / len(
    class_results
)

macro_recall = sum(
    item["recall"]
    for item in class_results
) / len(
    class_results
)

print(
    "Macro F1:",
    macro_f1
)

print(
    "Macro Recall:",
    macro_recall
)


# 53. Weighted Averaging

Weighted averaging weights each class by its support.

Conceptually:

$$
\boxed{
Metric_{weighted}
=
\frac{
\sum_i support_i\cdot Metric_i
}{
\sum_i support_i
}
}
$$

Large classes influence the result more.


In [ ]:
total_support = sum(
    item["support"]
    for item in class_results
)

weighted_f1 = sum(
    item["support"]
    * item["f1"]
    for item in class_results
) / total_support

print(
    "Weighted F1:",
    weighted_f1
)


# 54. Micro Averaging

Micro averaging aggregates decisions across classes before calculating the metric.

In standard single-label multi-class classification, micro precision, micro recall, micro F1, and accuracy are closely related and often equal.

Macro metrics are usually more sensitive to minority-class failure.


# 55. Macro vs Weighted Metrics

$$
\begin{array}{|c|c|}
\hline
\textbf{Macro} & \textbf{Weighted} \\
\hline
Equal\ weight\ per\ class & Weight\ by\ class\ support \\
\hline
Highlights\ minority\ classes & Reflects\ dataset\ frequency \\
\hline
Can\ fall\ sharply\ if\ one\ class\ fails & Can\ hide\ minority\ failure \\
\hline
\end{array}
$$


# 56. Balanced Accuracy

For binary classification, balanced accuracy is commonly:

$$
\boxed{
Balanced\ Accuracy
=
\frac{
Sensitivity+Specificity
}{
2
}
}
$$

For multi-class classification, it is commonly the average per-class recall.

This gives classes more equal influence than ordinary accuracy.


In [ ]:
balanced_accuracy = (
    macro_recall
)

print(
    "Balanced accuracy:",
    balanced_accuracy
)


# 57. Why Balanced Accuracy Helps

Suppose:

- Class 0 contains 90% of data
- Class 1 contains 10%

A classifier can have high ordinary accuracy while failing class 1.

Balanced accuracy gives each class equal importance through per-class recall.


# 58. Multi-Class ROC / AUROC

For multi-class classification, ROC analysis is often done using:

> **One-vs-Rest**

For class $k$:

- Class $k$ = positive
- Every other class = negative

Then compute a binary ROC curve for that class.


In [ ]:
multiclass_probabilities = torch.softmax(
    multiclass_logits,
    dim=1
)

class_index = 2

class_targets = (
    multiclass_targets
    == class_index
).long()

class_scores = (
    multiclass_probabilities[
        :,
        class_index
    ]
)

class_fpr, class_tpr, _ = (
    roc_curve_torch(
        class_targets,
        class_scores
    )
)

class_auroc = trapezoid_area(
    class_fpr,
    class_tpr
)

print(
    "Class 2 AUROC:",
    class_auroc
)


# 59. Macro Multi-Class AUROC

Compute one-vs-rest AUROC for each class, then average them.

$$
\boxed{
AUROC_{macro}
=
\frac{
1
}{
C
}
\sum_i AUROC_i
}
$$


In [ ]:
def multiclass_ovr_aurocs(
    targets,
    probabilities
):
    num_classes = (
        probabilities.shape[1]
    )

    values = []

    for class_index in range(
        num_classes
    ):
        binary_targets = (
            targets
            == class_index
        ).long()

        scores = probabilities[
            :,
            class_index
        ]

        fpr, tpr, _ = (
            roc_curve_torch(
                binary_targets,
                scores
            )
        )

        values.append(
            trapezoid_area(
                fpr,
                tpr
            )
        )

    return values

class_aurocs = (
    multiclass_ovr_aurocs(
        multiclass_targets,
        multiclass_probabilities
    )
)

print(
    "Per-class AUROCs:",
    class_aurocs
)

print(
    "Macro AUROC:",
    sum(class_aurocs)
    / len(class_aurocs)
)


# 60. Calibration Intuition

Classification performance is not only about ranking.

If a model predicts:

$$
0.9
$$

probability, we would like events predicted near 0.9 to occur roughly:

$$
90\%
$$

of the time.

This relationship between predicted confidence and observed frequency is:

> **Calibration**


# 61. Discrimination vs Calibration

These are different.

## Discrimination

Can the model rank/separate positives and negatives?

Metrics:

- AUROC
- AUPRC

## Calibration

Do predicted probabilities correspond to real event frequencies?

A model can have excellent AUROC but poorly calibrated probabilities.


# 62. Calibration Example

Suppose 100 cases are assigned predicted probability:

$$
0.8
$$

If the model is well calibrated, approximately:

$$
80
$$

of them should actually be positive.

Calibration matters when probabilities influence decisions.


# 63. Reliability Diagram

A reliability diagram:

1. Divides predictions into probability bins
2. Computes average predicted probability in each bin
3. Computes observed positive frequency in each bin
4. Compares the two


In [ ]:
torch.manual_seed(42)

calibration_probs = torch.rand(
    300
)

calibration_targets = torch.bernoulli(
    calibration_probs
).long()

print(
    calibration_probs.shape,
    calibration_targets.shape
)


# 64. Computing Calibration Bins


In [ ]:
def calibration_bins(
    probabilities,
    targets,
    num_bins=10
):
    edges = torch.linspace(
        0.0,
        1.0,
        num_bins + 1
    )

    mean_confidences = []
    observed_frequencies = []
    counts = []

    for index in range(
        num_bins
    ):
        lower = edges[index]
        upper = edges[index + 1]

        if index == (
            num_bins - 1
        ):
            mask = (
                (probabilities >= lower)
                & (probabilities <= upper)
            )
        else:
            mask = (
                (probabilities >= lower)
                & (probabilities < upper)
            )

        count = mask.sum().item()

        if count == 0:
            mean_confidences.append(
                float("nan")
            )

            observed_frequencies.append(
                float("nan")
            )

            counts.append(0)

            continue

        mean_confidences.append(
            probabilities[
                mask
            ].mean().item()
        )

        observed_frequencies.append(
            targets[
                mask
            ].float().mean().item()
        )

        counts.append(count)

    return (
        mean_confidences,
        observed_frequencies,
        counts
    )

mean_conf, observed_freq, counts = (
    calibration_bins(
        calibration_probs,
        calibration_targets,
        num_bins=10
    )
)

print(
    "Counts:",
    counts
)


# 65. Plotting a Reliability Diagram


In [ ]:
valid_x = []
valid_y = []

for confidence, frequency in zip(
    mean_conf,
    observed_freq
):
    if (
        not math.isnan(
            confidence
        )
        and
        not math.isnan(
            frequency
        )
    ):
        valid_x.append(
            confidence
        )

        valid_y.append(
            frequency
        )

plt.figure(figsize=(6, 6))

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Perfect calibration"
)

plt.plot(
    valid_x,
    valid_y,
    marker="o",
    label="Model"
)

plt.xlabel(
    "Mean Predicted Probability"
)

plt.ylabel(
    "Observed Positive Frequency"
)

plt.title(
    "Reliability Diagram"
)

plt.legend()
plt.show()


# 66. Expected Calibration Error Intuition

One simple calibration summary is based on the weighted difference between:

- Average confidence in each bin
- Observed frequency in each bin

A common form is:

$$
\boxed{
ECE
=
\sum_b
\frac{
n_b
}{
N
}
\left|
acc_b-conf_b
\right|
}
$$

Smaller is generally better.

ECE depends on binning choices, so it should not be treated as an absolute universal truth.


In [ ]:
def expected_calibration_error(
    probabilities,
    targets,
    num_bins=10
):
    mean_conf, observed, counts = (
        calibration_bins(
            probabilities,
            targets,
            num_bins=num_bins
        )
    )

    total = len(
        probabilities
    )

    ece = 0.0

    for confidence, frequency, count in zip(
        mean_conf,
        observed,
        counts
    ):
        if count == 0:
            continue

        ece += (
            count
            / total
            * abs(
                confidence
                - frequency
            )
        )

    return ece

ece = expected_calibration_error(
    calibration_probs,
    calibration_targets,
    num_bins=10
)

print(
    "ECE:",
    ece
)


# 67. Calibration Is Important in Medical Prediction

Suppose two patients receive predicted risks:

$$
0.20
$$

and:

$$
0.80
$$

If those numbers are used as estimated risk, they should have meaningful probabilistic interpretation.

This matters for:

- Risk communication
- Decision support
- Triage
- Treatment thresholds
- Shared decision making


# 68. Error Analysis

Metrics tell you **how much** error exists.

Error analysis tries to understand:

> **Why does the model fail?**

Useful error categories may include:

- Low-quality images
- Rare pathology
- Device/site differences
- Small lesions
- Ambiguous labels
- Atypical anatomy
- Poor acquisition
- Artifacts


# 69. High-Confidence Errors

One especially useful subset is:

> Incorrect predictions made with high confidence.

These cases may reveal:

- Leakage
- Label noise
- Distribution shift
- Systematic model bias
- Difficult subgroups


In [ ]:
example_targets = torch.tensor([
    1, 0, 1, 0, 1, 0
])

example_probs = torch.tensor([
    0.95,
    0.90,
    0.85,
    0.20,
    0.15,
    0.05
])

example_predictions = (
    example_probs
    >= 0.5
).long()

wrong = (
    example_predictions
    != example_targets
)

confidence = torch.maximum(
    example_probs,
    1.0 - example_probs
)

high_conf_wrong = (
    wrong
    & (confidence >= 0.8)
)

print(
    "High-confidence error indices:",
    torch.where(
        high_conf_wrong
    )[0]
)


# 70. Confidence Is Not Calibration

A model can be highly confident and wrong.

Confidence alone does not guarantee correctness.

This is why we need:

- Calibration analysis
- Error inspection
- Out-of-distribution evaluation


# 71. Subgroup Analysis

Overall performance can hide important subgroup failures.

Possible subgroups:

- Age groups
- Sex
- Disease severity
- Scanner manufacturer
- Hospital/site
- Probe type
- Acquisition quality

For each subgroup, consider reporting:

- Sample count
- Sensitivity
- Specificity
- AUROC
- AUPRC

when appropriate.


# 72. Site-Specific Analysis for Ultrasound

Ultrasound appearance can differ across:

- Scanner manufacturers
- Hospitals
- Operators
- Probes
- Acquisition protocols

A model may perform well overall while failing one site.

Always consider whether site/device subgroup evaluation is relevant to your research question.


# 73. Per-Patient vs Per-Image Evaluation

Medical imaging often contains multiple images per patient.

Per-image metrics treat every image as independent.

But the clinical decision may be per patient.

You must define the evaluation unit clearly:

$$
\boxed{
Image
\neq
Study
\neq
Patient
}
$$

The correct unit depends on the intended use.


# 74. Why Patient-Level Metrics Can Differ

Suppose one patient has 20 nearly identical images.

Counting all 20 equally can give that patient much more influence than a patient with only one image.

Patient-level aggregation may be more appropriate for some clinical tasks.


# 75. Choosing Metrics for Medical Imaging

Metric choice depends on the application.

## Screening

Missing disease may be very costly.

Often prioritize:

- Sensitivity / recall
- Negative-case tradeoff
- AUPRC when positives are rare

## Confirmatory Testing

False positives may be costly.

Specificity and precision can become more important.

## Risk Prediction

Calibration may be essential.


# 76. Sensitivity Is Not Always the Only Goal

A model with:

$$
100\%
$$

sensitivity can be useless if it predicts every patient as positive.

Then:

$$
Specificity=0
$$

Always consider the tradeoff.


# 77. Specificity Is Not Always the Only Goal

A model predicting everyone negative can achieve very high specificity in some datasets but:

$$
Sensitivity=0
$$

A clinically useful model needs an operating point aligned with the task.


# 78. Precision Depends on Prevalence

Precision is strongly affected by disease prevalence.

A model may have the same sensitivity and specificity in two populations but different precision because prevalence differs.

This is important when moving from:

- Research cohort
- To real clinical population


# 79. Demonstrating Prevalence Effects

Using fixed:

$$
Sensitivity=0.9
$$

and:

$$
Specificity=0.9
$$

we can calculate expected precision at different prevalence levels.


In [ ]:
def precision_from_operating_characteristics(
    sensitivity,
    specificity,
    prevalence
):
    numerator = (
        sensitivity
        * prevalence
    )

    denominator = (
        sensitivity
        * prevalence
        +
        (1.0 - specificity)
        * (1.0 - prevalence)
    )

    return (
        numerator
        / denominator
        if denominator > 0
        else 0.0
    )

for prevalence in [
    0.01,
    0.05,
    0.10,
    0.50
]:
    value = (
        precision_from_operating_characteristics(
            sensitivity=0.9,
            specificity=0.9,
            prevalence=prevalence
        )
    )

    print(
        f"Prevalence {prevalence:.2f} "
        f"-> Precision {value:.3f}"
    )


# 80. Accuracy Also Depends on Prevalence

Accuracy can be written as:

$$
\boxed{
Accuracy
=
Sensitivity\cdot Prevalence
+
Specificity\cdot(1-Prevalence)
}
$$

Therefore accuracy can change when class prevalence changes even if sensitivity and specificity remain unchanged.


# 81. Metric Confidence Intervals

A point estimate such as:

$$
AUROC=0.91
$$

does not tell us uncertainty.

For serious scientific reporting, confidence intervals are often important.

Possible methods include:

- Bootstrap confidence intervals
- Analytical intervals for some metrics
- Patient-level bootstrap when patients are the independent unit

We will keep this notebook focused on metric fundamentals, but uncertainty should not be ignored in research.


# 82. Why the Bootstrap Unit Matters

If multiple images come from one patient, bootstrapping individual images may underestimate uncertainty.

Often the resampling unit should match the independent unit:

> **Patient-level bootstrap**

when the study is patient-based.


# 83. Model Comparison Requires the Same Test Set

If Model A and Model B are evaluated on different test samples, differences may reflect dataset difficulty rather than model quality.

For a fair comparison:

- Same test set
- Same labels
- Same evaluation protocol
- Same metric definitions


# 84. Metrics Should Be Defined Before Final Testing

A good protocol defines:

- Primary metric
- Secondary metrics
- Threshold-selection rule
- Test set
- Subgroup analyses

before looking at final test performance.

This reduces metric cherry-picking.


# 85. Common Mistake — Reporting Only Accuracy

Especially with imbalance, always inspect:

- Confusion matrix
- Sensitivity
- Specificity
- Precision
- F1
- AUROC
- AUPRC

as appropriate.


# 86. Common Mistake — Choosing Threshold on Test Data

Threshold tuning is model development.

Use validation data.

Then freeze the threshold before final test evaluation.


# 87. Common Mistake — Applying the Default 0.5 Threshold Automatically

A threshold of:

$$
0.5
$$

is common, but not universally optimal.

The correct threshold depends on:

- Error costs
- Sensitivity requirement
- Specificity requirement
- Clinical workflow


# 88. Common Mistake — Treating AUROC as an Operating-Point Metric

AUROC summarizes ranking across many thresholds.

It does not tell you the actual sensitivity or specificity at the threshold you plan to deploy.

Always report operating-point metrics when deployment decisions depend on them.


# 89. Common Mistake — Ignoring AUPRC With Rare Positives

When positives are rare, AUROC may look strong even when positive predictions have poor precision.

AUPRC can provide useful complementary information.


# 90. Common Mistake — Comparing AUPRC Without Prevalence Context

AUPRC baseline depends on positive prevalence.

Always report or understand prevalence when interpreting AUPRC.


# 91. Common Mistake — Using Macro and Weighted Metrics Interchangeably

Macro:

> Equal importance to every class.

Weighted:

> Importance proportional to support.

They answer different questions.


# 92. Common Mistake — Ignoring Calibration

A model used only for ranking may not require perfectly calibrated probabilities.

But if predicted probabilities are interpreted as risk, calibration matters.


# 93. Common Mistake — Per-Image Split and Per-Image Evaluation With Patient Duplicates

If the same patient contributes many correlated images:

- Split design can leak information
- Standard errors can be too optimistic
- Per-image metrics may overrepresent some patients

Define the independent evaluation unit clearly.


# 94. Common Mistake — Looking Only at Aggregate Metrics

A strong overall AUROC can coexist with poor performance:

- At one site
- On one scanner
- In one subgroup
- On rare disease cases

Perform error and subgroup analysis where appropriate.


# 95. Practical Binary Evaluation Function

Let's create a utility that returns common threshold-based metrics.


In [ ]:
def evaluate_binary_classifier(
    targets,
    probabilities,
    threshold=0.5
):
    metrics = binary_metrics(
        targets,
        probabilities,
        threshold=threshold
    )

    fpr, tpr, _ = (
        roc_curve_torch(
            targets,
            probabilities
        )
    )

    precision_curve, recall_curve, _ = (
        precision_recall_curve_torch(
            targets,
            probabilities
        )
    )

    metrics["auroc"] = (
        trapezoid_area(
            fpr,
            tpr
        )
    )

    metrics["auprc"] = (
        trapezoid_area(
            recall_curve,
            precision_curve
        )
    )

    metrics["prevalence"] = (
        targets.float().mean().item()
    )

    return metrics

summary = (
    evaluate_binary_classifier(
        targets,
        probabilities,
        threshold=0.5
    )
)

for key, value in summary.items():
    print(
        key,
        ":",
        value
    )


# 96. Collecting Predictions From a PyTorch Model

A normal evaluation pipeline first collects:

- Targets
- Logits
- Probabilities

from the entire validation or test set.


In [ ]:
def collect_binary_outputs(
    model,
    loader,
    device
):
    model.eval()

    all_targets = []
    all_logits = []

    with torch.inference_mode():
        for inputs, targets in loader:
            inputs = inputs.to(
                device
            )

            logits = model(
                inputs
            ).squeeze(
                dim=1
            )

            all_targets.append(
                targets.cpu()
            )

            all_logits.append(
                logits.cpu()
            )

    targets = torch.cat(
        all_targets
    )

    logits = torch.cat(
        all_logits
    )

    probabilities = torch.sigmoid(
        logits
    )

    return (
        targets,
        logits,
        probabilities
    )


# 97. Why Collect First and Evaluate Second?

Separating prediction collection from metric calculation is useful because:

- Threshold can be changed without rerunning the model
- ROC/PR curves need all scores
- Calibration can be analyzed
- Predictions can be saved
- Error analysis becomes easier


# 98. Multi-Class Output Collection


In [ ]:
def collect_multiclass_outputs(
    model,
    loader,
    device
):
    model.eval()

    all_targets = []
    all_logits = []

    with torch.inference_mode():
        for inputs, targets in loader:
            inputs = inputs.to(
                device
            )

            logits = model(
                inputs
            )

            all_targets.append(
                targets.cpu()
            )

            all_logits.append(
                logits.cpu()
            )

    targets = torch.cat(
        all_targets
    )

    logits = torch.cat(
        all_logits
    )

    probabilities = torch.softmax(
        logits,
        dim=1
    )

    predictions = logits.argmax(
        dim=1
    )

    return (
        targets,
        logits,
        probabilities,
        predictions
    )


# 99. Saving Evaluation Outputs

For reproducible analysis, consider saving:

- Sample ID
- True label
- Logits
- Probabilities
- Predicted class
- Split
- Model/checkpoint ID

This allows metric calculations to be reproduced without rerunning inference.


# 100. Medical-Imaging Evaluation Checklist

Before reporting results, ask:

1. What is the independent unit — image, study, or patient?
2. Is the test set patient-independent?
3. What is disease prevalence?
4. What is the primary metric?
5. What threshold-selection rule was used?
6. Were threshold and hyperparameters chosen without test data?
7. What are sensitivity and specificity at the final threshold?
8. What is precision?
9. What are AUROC and AUPRC?
10. Are confidence intervals needed?
11. Are subgroup results needed?
12. Are there external-site results?
13. Is calibration relevant?
14. Have high-confidence errors been inspected?


# 101. Practice Exercises

Try these before looking at the solutions.

## Exercise 1

Given:

$$
TP=40,\ FP=10,\ TN=35,\ FN=15
$$

calculate:

- Accuracy
- Precision
- Recall
- Specificity
- F1

## Exercise 2

Create a binary prediction example and compute its confusion counts manually.

## Exercise 3

Compare metrics at thresholds:

$$
0.3,\ 0.5,\ 0.7
$$

## Exercise 4

Plot an ROC curve from probabilities and targets.

## Exercise 5

Calculate AUROC using the trapezoidal rule.

## Exercise 6

Plot a precision-recall curve.

## Exercise 7

Find the threshold that maximizes F1 on validation data.

## Exercise 8

Create a 3-class confusion matrix and calculate per-class recall.

## Exercise 9

Calculate macro F1 and weighted F1.

## Exercise 10

Create a simple reliability diagram.


# 102. Conceptual Challenges

## Challenge 1

Why can 95% accuracy be useless?

## Challenge 2

What is the difference between precision and sensitivity?

## Challenge 3

What does specificity measure?

## Challenge 4

Why does lowering the threshold usually increase sensitivity?

## Challenge 5

Why should threshold selection happen on validation data?

## Challenge 6

What does AUROC measure that threshold-specific sensitivity does not?

## Challenge 7

Why can AUPRC be especially useful with rare positives?

## Challenge 8

Why does AUPRC depend on prevalence?

## Challenge 9

What is the difference between macro and weighted averaging?

## Challenge 10

How can a model have good AUROC but poor calibration?

## Challenge 11

Why should medical-image evaluation sometimes be performed per patient rather than per image?

## Challenge 12

Why should subgroup and failure-case analysis accompany aggregate metrics?


# 103. Exercise Solutions


In [ ]:
# Exercise 1
tp_ex = 40
fp_ex = 10
tn_ex = 35
fn_ex = 15

accuracy_ex = (
    (tp_ex + tn_ex)
    / (
        tp_ex
        + fp_ex
        + tn_ex
        + fn_ex
    )
)

precision_ex = (
    tp_ex
    / (tp_ex + fp_ex)
)

recall_ex = (
    tp_ex
    / (tp_ex + fn_ex)
)

specificity_ex = (
    tn_ex
    / (tn_ex + fp_ex)
)

f1_ex = (
    2
    * precision_ex
    * recall_ex
    / (
        precision_ex
        + recall_ex
    )
)

print(
    "Accuracy:",
    accuracy_ex
)

print(
    "Precision:",
    precision_ex
)

print(
    "Recall:",
    recall_ex
)

print(
    "Specificity:",
    specificity_ex
)

print(
    "F1:",
    f1_ex
)


In [ ]:
# Exercise 2 and 3
exercise_targets = torch.tensor([
    1, 1, 1, 0, 0, 0
])

exercise_probs = torch.tensor([
    0.9,
    0.6,
    0.4,
    0.8,
    0.3,
    0.1
])

for threshold in [
    0.3,
    0.5,
    0.7
]:
    print(
        binary_metrics(
            exercise_targets,
            exercise_probs,
            threshold=threshold
        )
    )


In [ ]:
# Exercise 4 and 5
exercise_fpr, exercise_tpr, _ = (
    roc_curve_torch(
        exercise_targets,
        exercise_probs
    )
)

exercise_auroc = trapezoid_area(
    exercise_fpr,
    exercise_tpr
)

print(
    "Exercise AUROC:",
    exercise_auroc
)


In [ ]:
# Exercise 6 and 7
exercise_precision, exercise_recall, _ = (
    precision_recall_curve_torch(
        exercise_targets,
        exercise_probs
    )
)

best_threshold = None
best_f1 = -1.0

for threshold in torch.linspace(
    0.0,
    1.0,
    101
):
    result = binary_metrics(
        exercise_targets,
        exercise_probs,
        threshold=float(
            threshold
        )
    )

    if result["f1"] > best_f1:
        best_f1 = result["f1"]
        best_threshold = float(
            threshold
        )

print(
    "Best threshold:",
    best_threshold
)

print(
    "Best F1:",
    best_f1
)


In [ ]:
# Exercise 8 and 9
exercise_targets_multi = torch.tensor([
    0, 0, 1, 1, 2, 2, 2
])

exercise_predictions_multi = torch.tensor([
    0, 1, 1, 1, 2, 0, 2
])

exercise_confusion = (
    confusion_matrix_multiclass(
        exercise_targets_multi,
        exercise_predictions_multi,
        num_classes=3
    )
)

exercise_results = (
    per_class_metrics(
        exercise_confusion
    )
)

macro_f1_ex = sum(
    item["f1"]
    for item in exercise_results
) / len(
    exercise_results
)

support_total = sum(
    item["support"]
    for item in exercise_results
)

weighted_f1_ex = sum(
    item["f1"]
    * item["support"]
    for item in exercise_results
) / support_total

print(
    exercise_confusion
)

print(
    "Macro F1:",
    macro_f1_ex
)

print(
    "Weighted F1:",
    weighted_f1_ex
)


# 104. Key Takeaways

In this notebook, we learned:

- Why accuracy is not enough
- Binary confusion matrices
- True positives
- False positives
- True negatives
- False negatives
- Precision
- Recall / sensitivity
- Specificity
- F1 score
- Class imbalance
- Threshold effects
- ROC curves
- False-positive rate
- AUROC
- Precision-recall curves
- AUPRC
- Prevalence effects
- Threshold selection
- Sensitivity-constrained thresholding
- Multi-class confusion matrices
- One-vs-rest analysis
- Per-class metrics
- Macro averaging
- Weighted averaging
- Micro averaging
- Balanced accuracy
- Multi-class AUROC
- Calibration
- Reliability diagrams
- ECE intuition
- Error analysis
- High-confidence errors
- Subgroup analysis
- Per-image vs per-patient evaluation
- Metric selection for medical imaging
- Common evaluation mistakes

The central binary confusion matrix is:

$$
\boxed{
\begin{array}{c|c|c}
 & Predicted\ + & Predicted\ - \\
\hline
Actual\ + & TP & FN \\
\hline
Actual\ - & FP & TN \\
\end{array}
}
$$

The most important threshold-dependent metrics are:

$$
\boxed{
Precision
=
\frac{TP}{TP+FP}
}
$$

$$
\boxed{
Sensitivity
=
\frac{TP}{TP+FN}
}
$$

$$
\boxed{
Specificity
=
\frac{TN}{TN+FP}
}
$$

And the most important evaluation principle is:

$$
\boxed{
\text{Metric Choice}
\rightarrow
\text{Must Match the Real Objective}
}
$$


# 105. Check Your Understanding

Before moving forward, make sure you can answer these without searching:

1. Why can accuracy be misleading?
2. What is a true positive?
3. What is a false positive?
4. What is a true negative?
5. What is a false negative?
6. What does precision measure?
7. What does recall/sensitivity measure?
8. What does specificity measure?
9. What does F1 combine?
10. Why does threshold choice matter?
11. Why should thresholds be selected on validation data?
12. What are TPR and FPR?
13. What does an ROC curve show?
14. What does AUROC summarize?
15. Why is AUROC considered a ranking metric?
16. What does a precision-recall curve show?
17. Why is AUPRC useful for imbalanced data?
18. Why does AUPRC baseline depend on prevalence?
19. What is one-vs-rest evaluation?
20. What is macro averaging?
21. What is weighted averaging?
22. What is balanced accuracy?
23. What is calibration?
24. How can discrimination and calibration differ?
25. What is a reliability diagram?
26. Why inspect high-confidence errors?
27. Why perform subgroup analysis?
28. Why can per-patient evaluation differ from per-image evaluation?
29. Why does precision change with prevalence?
30. Which metrics might be especially important for a medical screening task?


# Next Notebook

# 22 — Explainability and Model Interpretation for Vision Models

In the next notebook, we will study:

- Why model interpretation matters
- Feature-map visualization
- CNN activation maps
- Saliency maps
- Input gradients
- Grad-CAM intuition
- Implementing Grad-CAM
- Interpreting heatmaps carefully
- Occlusion sensitivity
- Confidence vs explanation
- Failure-case inspection
- Shortcut learning
- Spurious correlations
- Explainability pitfalls
- Ultrasound-specific interpretation concerns
- Using explanations as debugging tools, not proof of causality
